<a href="https://colab.research.google.com/github/EHSANHAZARI/Uganda-Crop-Disaese/blob/main/AIProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [40]:
import os
import torch
from torchvision import datasets, transforms
from google.colab import drive

# --- 1. MOUNT GOOGLE DRIVE ---
# Google Colab environments are temporary. We mount your Google Drive
# so we can access your safely stored, permanent files.
print("1. Connecting to permanent Google Drive...")
drive.mount('/content/drive')

# --- 2. LOCAL STORAGE SPEED BOOST ---
# Colab's local storage (/content/) is much faster than reading directly
# from Google Drive over the network. We set the path to the fast local drive here.
print("2. Checking if Colab deleted our temporary files...")
data_dir = "/content/Data/train"

# os.path.exists checks if the files are already in the fast local storage.
# If they are missing (because Colab reset), it copies them over.
if not os.path.exists(data_dir):
    print("Files missing! Copying from Google Drive back to fast local storage... (Wait 1-2 mins)")
    # The '!cp' command copies your entire Data folder into Colab's high-speed memory.
    !cp -r /content/drive/MyDrive/Data /content/Data
    print("✅ Copy complete!")
else:
    # If the files are already there, we skip the 2-minute wait!
    print("✅ Files are already here!")

print("3. Preparing the dataset...")

# --- 3. DATA TRANSFORMATION PIPELINE ---
# This custom class ensures every single image has 3 color channels (Red, Green, Blue).
# If a black-and-white image sneaks into the dataset, it converts it so it doesn't crash the model.
class ConvertToRGB(object):
    def __call__(self, img):
        if img.mode != "RGB":
            img = img.convert("RGB")
        return img

# transforms.Compose chains our image processing steps together in order:
# 1. Convert to RGB (Color check)
# 2. Resize to 224x224 (Standard size required for most neural networks, including ResNet50)
# 3. ToTensor: Converts raw pixels (0-255) into PyTorch math tensors (0.0 to 1.0)
# 4. Normalize: Shifts the color values to a standard mathematical range that helps the AI learn faster.
transform_normalized = transforms.Compose([
    ConvertToRGB(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 4. DATASET LOADING ---
# ImageFolder automatically looks at your subfolders and uses their names as the category labels.
try:
    normalized_dataset = datasets.ImageFolder(root=data_dir, transform=transform_normalized)
    print(f"🎉 SUCCESS! Ready to train with {len(normalized_dataset)} images.")
except Exception as e:
    # If something goes wrong (like a typo in the file path), this catches the error and prints it clearly.
    print("❌ Error loading data:")
    print(e)

1. Connecting to permanent Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2. Checking if Colab deleted our temporary files...
✅ Files are already here!
3. Preparing the dataset...
🎉 SUCCESS! Ready to train with 14932 images.


In [41]:
import os  #It helps with 1.Looking for folders 2.Creating Files 3.Deleting Files 4.Checking path to your computer
import time  #gives Python tools to pause execution, get the current time, and measure how long operations take.
import torch  #This is helpful for trainning AI engine loads the PyTorch library, which provides tensors, fast math operations, and tools for building and training deep learning models
import torch.nn as nn  #Helps to build Neural Netwrok imports PyTorch’s neural network module and gives it the short name nn so we can easily build neural network layers.
import torch.nn.functional as F  #provides individual neural network functions like ReLU, softmax, and loss calculations
import torch.optim as optim  #provides algorithms that update model weights to reduce prediction errors during training.
from torchvision import datasets, transforms, models  #dataset Ready-made training data   Transforms->Transforms modify images before training.
from torch.utils.data import DataLoader, random_split #DataLoader feeds data to the model in batches -- random_split divides datasets into training and validation sets.
from tqdm.notebook import tqdm  #Shows the model processing



In [42]:
from google.colab import drive  # Our data is located at the google drive as the google colab has temporary file storage
drive.mount('/content/drive')

# Set the device to CPU  -- GPU is faster than CPU in processing the images and specialized in doing math Can do thousands of calcuations and perfect for deep learning
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device.")

# 3. Set your data directory
# Pointing to the high-speed local Colab storage!
data_dir = "/content/Data/train"
print("Data Directory:", data_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using cuda device.
Data Directory: /content/Data/train


In [43]:
#This class create a neurl network architecture -- Our class is extended from nn.Module  -- nn.Module alraedy know how to store layers , track parameters , move model to gpu , run forward passes , save load models
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=5):  #The model predict 5 categories 4 diesaese and 1 helthy
        super(SimpleCNN, self).__init__()  #This is initilizing everything from the nn.Module parent class -- nn.Module is the parent class.
        # 1st Convolutional Layer
        #Convolution layer is the layer where scan the images with small filters to detect patterns and produce feature maps used for image recognition
        #Using Conv2d because it has two dimension here heigh and width
        #in_channels = 3 tells how many channels the input image has here is Red Green Blue
        #out_channels=16 tells how many filters it will learn filter 1-> vertical edges , filter 2 -> horizental edges and ...
        #kernel_size=3 this defines the size of the filter the filters slides across the images and check small region --> (3 * 3 magnifying glass)
        #padding = 1 Padding preserves the image size after convolution.
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        #Max Pooling is looking at small regions and keeping only the stronges signal
        # Kernel size = 2 2 * 2 find the biggest number
        #Stride says what is the next step would be if its [0-1] ->[2-3]->[4-5]
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # 2nd Convolutional Layer
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)

        # Fully Connected Layers
        #Flattening the images - convert 3d images to one long vector
        self.fc1 = nn.Linear(32 * 56 * 56, 128)    # 32 * 56 * 56 input -- 128 output
        self.fc2 = nn.Linear(128, num_classes)  # conver the 128 outoput of previous layer to 5 disease classes

        #Forward function sends the entry images through the machines in this exact order
        #Defines how data flows through the network
        # Relu ->Rectified Linear Unit --> its activation function that repalces negative values with zero allowing neural network to learn complex nonlinear patterns
        #Activation function transfroms a neuron's output using a nonlinear rule so the neural network can learn complex pattern

    def forward(self, x):
      x = self.pool(F.relu(self.conv1(x)))  #self.pool shrink the image while keeping the most importnat vlaues
      x = self.pool(F.relu(self.conv2(x)))
      x = x.view(-1, 32 * 56 * 56) # Flatten
      x = F.relu(self.fc1(x))
      x = self.fc2(x)
      return x

# Test if the model builds and can be moved to the GPU
model_simple = SimpleCNN().to(device)
print("Method 2 (Simple CNN) built successfully!")

Method 2 (Simple CNN) built successfully!


In [44]:
from tqdm.notebook import tqdm

#Train_epoch teaches the neural network by processing all trainning batches once, calculating the error and updating the model weight
#Model is the neural network we want to train
# loss_fn calcutes how wrong the prediction is : example nn.CrossEntropyLoss()
#Data_loader provied the datset in the batches
# device tells the model where to run
def train_epoch(model, optimizer, loss_fn, data_loader, device="cpu"):
    model.train()
    training_loss = 0.0
    # Going through the datasets batch by batch get the image and the correct label for it
    # input --> image , output --> correct label
    # We wrap the data_loader in tqdm to see a progress bar for every batch
    for inputs, targets in tqdm(data_loader, desc="Training Batches", leave=False):
      #we have to make sure input and target are both on the same machine that's what .to(device) do
        inputs, targets = inputs.to(device), targets.to(device)
        #.zero_grad clears the stored gradients(mistakes) before calculating new ones.
        optimizer.zero_grad()
        #Send the input data (images) into the neural network and get the prediction.
        output = model(inputs)
        # Calcuating the loss using the output and label (targets)
        loss = loss_fn(output, targets)
        #Compute gradients   loss.backward()-> Which weights caused this mistake?
        loss.backward()
        #Update weights
        optimizer.step()
        #loss.data.item() --> This converts the tensor into a normal Python number
        #inputs.size(0) --> gives the batch size
        training_loss += loss.data.item() * inputs.size(0)
    #Finding the average mistake and return it
    return training_loss / len(data_loader.dataset)

#score function measures how good the model performs on dataset
#model -> neural netowrk
#data_loader->dataset batches
#loss_fn --> loss function
def score(model, data_loader, loss_fn, device="cpu"):
  #swtich the model to evaluation mode model.train() → training mode  model.eval() → testing mode
    model.eval()
    #this variable calcuate the loss from all batches
    total_loss = 0
    #this counts how many predictions are right
    total_correct = 0
    #n_observations counts how many sample we evaluated
    n_observations = 0
    #Disable Gradient Tracking ->Do NOT calculate gradients , Do NOT update weights , Just make predictions
    with torch.no_grad():
        # We add tqdm here too so you can watch the validation scoring progress!
        for inputs, targets in tqdm(data_loader, desc="Scoring Batches", leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            output = model(inputs)

            loss = loss_fn(output, targets)
            total_loss += loss.data.item() * inputs.size(0)
            #Predicted class vs True class , comparing target with the prediction
            correct = torch.eq(torch.argmax(output, dim=1), targets)
            #total_correct += torch.sum(correct).item()
            total_correct += torch.sum(correct).item()
            #This counts how many images were evaluated.
            n_observations += inputs.size(0)
    return total_loss / n_observations, total_correct / n_observations

In [45]:
from sklearn.model_selection import KFold
from torch.utils.data import SubsetRandomSampler
import time

#run_kfold_experiment() performs K-Fold cross-validation by repeatedly training and evaluating
#the model on different splits of the dataset to obtain a reliable performance estimate.
def run_kfold_experiment(dataset, model_class, batch_size=32, lr=0.001, k_folds=3, epochs=5, device="cuda"):
    # Initialize the K-Fold splitter
    #Shuffle the dataset to have random data -> Prevents bias in data order
    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

    #fold_results-> store the result of each folder
    fold_results = []

    #recording the start time of experiment
    total_start_time = time.time()

    print(f"Starting {k_folds}-Fold Cross Validation...")
    print("-" * 30)

    #Loop through each folder
    #kfold.split(dataset) produces training and validation indices.
    for fold, (train_ids, val_ids) in enumerate(kfold.split(dataset)):
        print(f"FOLD {fold + 1} initialized. Training starting...")

        # Create data samplers for this specific fold
        train_subsampler = SubsetRandomSampler(train_ids)
        val_subsampler = SubsetRandomSampler(val_ids)

        # --- HERE ARE THE CHANGES ---
        # DataLoaders
        # Added num_workers=2: Uses 2 background CPU threads to fetch the next images from Google Drive ahead of time.
        # Added pin_memory=True: Creates a fast-lane to transfer those fetched images directly into the GPU's memory.
        train_loader = DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=train_subsampler,
            num_workers=2,         # NEW
            pin_memory=True        # NEW
        )
        val_loader = DataLoader(
            dataset,
            batch_size=batch_size,
            sampler=val_subsampler,
            num_workers=2,         # NEW
            pin_memory=True        # NEW
        )
        # -----------------------------

        # Initialize a fresh model and optimizer for this fold
        model = model_class().to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.CrossEntropyLoss()

        fold_start_time = time.time()

        # Train for the specified number of epochs
        for epoch in range(1, epochs + 1):
            epoch_start = time.time()
            train_epoch(model, optimizer, loss_fn, train_loader, device)
            epoch_duration = time.time() - epoch_start

            # Progress tracking added here
            print(f"  -> Completed Epoch {epoch}/{epochs} (Took {epoch_duration:.2f} seconds)")

        # Score the fold on the validation split
        val_loss, val_acc = score(model, val_loader, loss_fn, device)
        fold_time = time.time() - fold_start_time

        print(f"Validation Accuracy: {val_acc*100:.2f}% | Time Taken: {fold_time:.2f} seconds\n")
        fold_results.append({'fold': fold + 1, 'accuracy': val_acc, 'time': fold_time})

    # Calculate overall metrics
    total_time = time.time() - total_start_time
    avg_acc = sum([res['accuracy'] for res in fold_results]) / k_folds

    print("=" * 30)
    print(f"EXPERIMENT COMPLETE")
    print(f"Average Accuracy: {avg_acc*100:.2f}%")
    print(f"Total Computation Time: {total_time:.2f} seconds")

    return fold_results

In [46]:
# Testing Method 2 with Learning Rate = 0.001 and Batch Size = 32
print("--- RUNNING EXPERIMENT: Simple CNN ---")
simple_results = run_kfold_experiment(
    dataset=normalized_dataset,
    model_class=SimpleCNN,
    batch_size=32,
    lr=0.001,
    k_folds=3,
    epochs=5,
    device=device
)

--- RUNNING EXPERIMENT: Simple CNN ---
Starting 3-Fold Cross Validation...
------------------------------
FOLD 1 initialized. Training starting...


Training Batches:   0%|          | 0/312 [00:00<?, ?it/s]

  -> Completed Epoch 1/5 (Took 73.71 seconds)


Training Batches:   0%|          | 0/312 [00:00<?, ?it/s]

  -> Completed Epoch 2/5 (Took 70.09 seconds)


...

  -> Completed Epoch 4/5 (Took 69.46 seconds)


Training Batches:   0%|          | 0/312 [00:00<?, ?it/s]

  -> Completed Epoch 5/5 (Took 67.52 seconds)


Scoring Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Validation Accuracy: 41.33% | Time Taken: 376.18 seconds

EXPERIMENT COMPLETE
Average Accuracy: 41.27%
Total Computation Time: 1135.59 seconds


In [49]:
import torch.nn as nn
from torchvision import models

# Method 2: ResNet50 (Transfer Learning)
# We wrap ResNet50 inside a class so it plugs perfectly into your run_kfold_experiment function
class CustomResNet50(nn.Module):
    def __init__(self, num_classes=5): # We have 5 categories (4 diseases + 1 healthy)
        super(CustomResNet50, self).__init__()

        # 1. Load the pre-trained ResNet50 model
        # pretrained=True downloads weights that have already learned to recognize shapes, edges, and patterns
        # This saves us hundreds of hours of training time!
        self.resnet = models.resnet50(pretrained=True)

        # 2. Modify the final classification layer
        # The original ResNet50 was built to predict 1000 different ImageNet categories (dogs, cars, cats, etc.)
        # .in_features finds out exactly how many connections are flowing into that final layer (it is 2048)
        num_ftrs = self.resnet.fc.in_features

        # We replace the original 1000-output layer with a brand new nn.Linear layer that outputs exactly 5 classes
        self.resnet.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        # Pass the image through the entire modified ResNet50 network to get the final prediction
        return self.resnet(x)

# Print a success message to verify it built correctly
print("Method 1 (ResNet50) built successfully and is ready to train!")

Method 1 (ResNet50) built successfully and is ready to train!


In [50]:
# Testing Method 1 (ResNet50) with Learning Rate = 0.001 and Batch Size = 32
print("--- RUNNING EXPERIMENT: ResNet50 ---")
resnet_results = run_kfold_experiment(
    dataset=normalized_dataset,
    model_class=CustomResNet50,  # <-- We swapped SimpleCNN out for ResNet50 here!
    batch_size=32,
    lr=0.001,
    k_folds=3,
    epochs=5,
    device=device
)

--- RUNNING EXPERIMENT: ResNet50 ---
Starting 3-Fold Cross Validation...
------------------------------
FOLD 1 initialized. Training starting...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 133MB/s]


Training Batches:   0%|          | 0/312 [00:00<?, ?it/s]

...

  -> Completed Epoch 4/5 (Took 111.29 seconds)


Training Batches:   0%|          | 0/312 [00:00<?, ?it/s]

  -> Completed Epoch 5/5 (Took 110.58 seconds)


Scoring Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Validation Accuracy: 61.54% | Time Taken: 593.36 seconds

EXPERIMENT COMPLETE
Average Accuracy: 65.41%
Total Computation Time: 1784.44 seconds


In [66]:
!git init

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
fatal: unable to get current working directory: Transport endpoint is not connected


In [67]:
!git status



shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
fatal: Unable to read current working directory: Transport endpoint is not connected
